# Quantum Meta-RL for Anomaly Detection: Data Processing and Feature Extraction

This notebook focuses on the initial stages of the Quantum Meta-Reinforcement Learning pipeline:
1. **Data Gathering**: Loading multiple datasets (Breast Cancer, Ionosphere, Credit Card, NSL-KDD).
2. **Data Processing**: Scaling, Dimensionality Reduction (PCA), and Normalization for Quantum Circuits.
3. **Feature Extraction**: Encoding classical features into quantum states using a Parametrized Quantum Circuit (PQC).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import pennylane as qml
from pennylane import numpy as pnp

# Set random seed for reproducibility
np.random.seed(42)

## 1. Data Gathering

We implement functions to load various anomaly detection datasets from Scikit-learn and OpenML.

In [ ]:
def load_breast_cancer_dataset():
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    # target: 0 is malignant (anomaly), 1 is benign (normal) -> Map 0 to normal, 1 to anomaly
    df['target'] = 1 - data.target
    return df

def load_ionosphere_dataset():
    data = fetch_openml(data_id=59, as_frame=True, parser='auto')
    df = data.frame
    df['target'] = df['class'].apply(lambda x: 0 if x == 'g' else 1)
    df = df.drop(columns=['class'])
    return df

def load_credit_card_dataset():
    data = fetch_openml(data_id=1597, as_frame=True, parser='auto')
    df = data.frame
    df['target'] = df['Class'].astype(int)
    df = df.drop(columns=['Class'])
    return df

def load_nsl_kdd_dataset():
    data = fetch_openml(data_id=40685, as_frame=True, parser='auto')
    df = data.frame
    df['target'] = df['class'].apply(lambda x: 0 if str(x) == '1' else 1)
    df = df.drop(columns=['class'])
    df_numeric = df.select_dtypes(include=[np.number])
    return df_numeric

## 2. Data Processing

The `DatasetManager` class handles scaling, PCA (to fit quantum hardware constraints), and normalization to `[-pi, pi]` for angle encoding.

In [ ]:
class DatasetManager:
    def __init__(self, n_qubits=8):
        self.n_qubits = n_qubits
        self.datasets = {}
        self.scalers = {}
        self.pcas = {}

    def load_all(self):
        print("Loading Breast Cancer dataset...")
        self.datasets['breast_cancer'] = load_breast_cancer_dataset()
        print("Loading Ionosphere dataset...")
        self.datasets['ionosphere'] = load_ionosphere_dataset()
        # Loading small subsets for demonstration if full datasets are too large
        print("Data loading complete.")

    def preprocess(self, name):
        df = self.datasets[name]
        X = df.drop(columns=['target']).values
        y = df['target'].values

        X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        self.scalers[name] = scaler

        pca = PCA(n_components=self.n_qubits)
        X_train = pca.fit_transform(X_train)
        X_val = pca.transform(X_val)
        X_test = pca.transform(X_test)
        self.pcas[name] = pca

        def normalize_features(data):
            min_val = np.min(data, axis=0)
            max_val = np.max(data, axis=0)
            range_val = max_val - min_val
            range_val[range_val == 0] = 1
            return 2 * np.pi * (data - min_val) / range_val - np.pi

        X_train = normalize_features(X_train)
        X_val = normalize_features(X_val)
        X_test = normalize_features(X_test)

        return (X_train, y_train), (X_val, y_val), (X_test, y_test)

manager = DatasetManager(n_qubits=4)
manager.load_all()
(X_train, y_train), _, _ = manager.preprocess('breast_cancer')
print(f"Processed Breast Cancer shape: {X_train.shape}")

## 3. Feature Extraction (Quantum Encoding)

We use a Parametrized Quantum Circuit (PQC) to encode classical features into a high-dimensional Hilbert space.

In [ ]:
class PQCEncoder:
    def __init__(self, n_qubits=4, n_layers=1):
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.dev = qml.device("default.qubit", wires=n_qubits)
        self.params = pnp.random.uniform(low=-np.pi, high=np.pi, size=(n_layers, n_qubits, 2), requires_grad=True)
        
        @qml.qnode(self.dev)
        def circuit(inputs, params):
            # Angle Encoding (Data Embedding)
            for i in range(self.n_qubits):
                qml.RY(inputs[i], wires=i)
            
            # Variational Layers (Feature Transformation)
            for l in range(self.n_layers):
                for i in range(self.n_qubits):
                    qml.RY(params[l, i, 0], wires=i)
                    qml.RZ(params[l, i, 1], wires=i)
                
                for i in range(self.n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                qml.CNOT(wires=[self.n_qubits - 1, 0])
            
            return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]
        
        self.qnode = circuit

    def forward(self, x):
        if x.ndim == 1:
            return np.array(self.qnode(x, self.params))
        else:
            return np.array([self.qnode(sample, self.params) for sample in x])

pqc = PQCEncoder(n_qubits=4, n_layers=1)
quantum_features = pqc.forward(X_train[:5])
print("Original features (first 5 samples):\n", X_train[:5, :2])
print("Quantum features (first 5 samples):\n", quantum_features[:5, :2])

## 4. Visualizations

Visualizing the processed features and the quantum circuit.

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.scatterplot(x=X_train[:, 0], y=X_train[:, 1], hue=y_train, palette='viridis')
plt.title("Classical Features (PCA 1 vs PCA 2)")

plt.subplot(1, 2, 2)
all_quantum_features = pqc.forward(X_train[:100])
sns.scatterplot(x=all_quantum_features[:, 0], y=all_quantum_features[:, 1], hue=y_train[:100], palette='magma')
plt.title("Quantum Features (Expectation Values)")
plt.tight_layout()
plt.show()

In [ ]:
print("Quantum Circuit Diagram:")
print(qml.draw(pqc.qnode)(X_train[0], pqc.params))